# Session 8 (Colab) — Real FlashAttention-2 CUDA Kernel

Run this on a Colab GPU runtime (Runtime > Change runtime type > GPU).
Locally (Mac CPU/MPS) `notebooks/08_flashattention.ipynb` teaches backend *selection* via `sdpa_kernel`; this notebook exercises the actual fused FlashAttention-2 CUDA kernel and measures its speedup for real.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn.attention import SDPBackend, sdpa_kernel

assert torch.cuda.is_available(), 'Enable a GPU runtime: Runtime > Change runtime type > GPU'
device = torch.device('cuda')

In [ ]:
import time

def bench(backend, q, k, v, iters=20):
    with sdpa_kernel(backend):
        for _ in range(3):
            F.scaled_dot_product_attention(q, k, v, is_causal=True)
        torch.cuda.synchronize()
        start = time.perf_counter()
        for _ in range(iters):
            F.scaled_dot_product_attention(q, k, v, is_causal=True)
        torch.cuda.synchronize()
        return (time.perf_counter() - start) / iters

q = k = v = torch.randn(4, 16, 2048, 64, device=device, dtype=torch.float16)
math_t = bench(SDPBackend.MATH, q, k, v)
flash_t = bench(SDPBackend.FLASH_ATTENTION, q, k, v)
print(f'math backend:  {math_t*1000:.3f} ms/iter')
print(f'flash backend: {flash_t*1000:.3f} ms/iter')
print(f'speedup: {math_t/flash_t:.2f}x')

## Takeaway

FlashAttention's speedup grows with sequence length (its whole point is avoiding materializing the seq x seq attention matrix in HBM). Try increasing seq_len above and watch the gap widen.